In [1]:
# Python libs
import pandas as pd

# Magics
from helpers import (
    load_sql_magic,
)
load_sql_magic()          # %%sql   — query DataFrames via duckdb (no extra installs)

True

In [2]:
import duckdb

DB_PATH = "./data/ab_events.duckdb"

con = duckdb.connect(DB_PATH, read_only=True)
event_log = con.execute("SELECT * FROM events").df()
con.close()

print(event_log.shape)
event_log.head()

(2128519, 7)


,date,user_id,hash_id,country,experiment,event_type,amount
0,2025-01-01 21:01:12,5,7674613650421074157,US,"{""num01"":""a""}",page_view,NaN
1,2025-01-01 21:02:09,5,7674613650421074157,US,"{""num01"":""a""}",page_view,NaN
2,2025-01-01 14:45:14,6,1310192797669293303,GB,"{""num01"":""a""}",page_view,NaN
3,2025-01-01 14:46:03,6,1310192797669293303,GB,"{""num01"":""a""}",page_view,NaN
4,2025-01-01 22:36:08,7,1750302349509622455,US,"{""num01"":""a""}",page_view,NaN


In [3]:
with duckdb.connect(DB_PATH, read_only=True) as con:
    df = con.execute("SHOW tables").df()

print(df)

                     name
0                  events
1    fct_ab_buckets_daily
2  int_ab_events_bucketed
3           stg_event_log


In [4]:
%%sql

SELECT * FROM event_log


,date,user_id,hash_id,country,experiment,event_type,amount
0,2025-01-01 21:01:12,5,7674613650421074157,US,"{""num01"":""a""}",page_view,NaN
1,2025-01-01 21:02:09,5,7674613650421074157,US,"{""num01"":""a""}",page_view,NaN
2,2025-01-01 14:45:14,6,1310192797669293303,GB,"{""num01"":""a""}",page_view,NaN
3,2025-01-01 14:46:03,6,1310192797669293303,GB,"{""num01"":""a""}",page_view,NaN
4,2025-01-01 22:36:08,7,1750302349509622455,US,"{""num01"":""a""}",page_view,NaN
...,...,...,...,...,...,...,...
2128514,2025-03-03 11:00:44,24996,968404688415237291,US,"{""num01"":""a""}",page_view,NaN
2128515,2025-03-03 11:01:37,24996,968404688415237291,US,"{""num01"":""a""}",page_view,NaN
2128516,2025-03-03 19:46:05,24996,968404688415237291,US,"{""num01"":""a""}",page_view,NaN
2128517,2025-03-03 19:47:57,24996,968404688415237291,US,"{""num01"":""a""}",watch,NaN


In [5]:
with duckdb.connect(DB_PATH, read_only=True) as con:
    fct_ab_buckets_daily = con.execute("SELECT * FROM fct_ab_buckets_daily").df()


In [6]:
%%sql buckets <<

SELECT * FROM fct_ab_buckets_daily


,date_day,country,bucket,experiment_number,experiment_group,total_events,page_view_count,watch_count,add_to_cart_count,purchase_count,total_unique_users,u_page_view,u_watch,u_add_to_cart,u_purchase,purchase_amount
0,2025-01-06,US,93,num01,b,42,35,7,0,0,7,7,6,0,0,0.00
1,2025-01-06,US,47,num01,b,30,24,4,1,1,5,5,3,1,1,24.65
2,2025-01-06,US,118,num01,a,29,23,4,1,1,6,6,4,1,1,30.56
3,2025-01-06,US,194,num01,b,68,55,11,1,1,14,14,11,1,1,92.26
4,2025-01-06,US,161,num01,b,54,43,8,1,2,8,8,7,1,2,329.27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72722,2025-03-03,DE,14,num01,a,16,12,2,1,1,3,3,2,1,1,32.96
72723,2025-03-03,DE,197,num01,a,5,4,1,0,0,2,2,1,0,0,0.00
72724,2025-03-03,DE,168,num01,a,11,9,1,0,1,1,1,1,0,1,31.22
72725,2025-03-03,GB,33,num01,b,6,5,1,0,0,1,1,1,0,0,0.00


In [7]:
buckets[
    (buckets.date_day=="2025-01-10")
    & (buckets.country=='US')
]

,date_day,country,bucket,experiment_number,experiment_group,total_events,page_view_count,watch_count,add_to_cart_count,purchase_count,total_unique_users,u_page_view,u_watch,u_add_to_cart,u_purchase,purchase_amount
498,2025-01-10,US,138,num01,b,57,44,12,1,0,14,14,11,1,0,0.00
499,2025-01-10,US,40,num01,a,26,20,5,1,0,5,5,4,1,0,0.00
502,2025-01-10,US,197,num01,b,86,73,13,0,0,17,17,13,0,0,0.00
503,2025-01-10,US,59,num01,a,70,59,8,2,1,11,11,6,2,1,22.42
505,2025-01-10,US,176,num01,b,42,32,6,3,1,9,9,6,3,1,13.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64146,2025-01-10,US,170,num01,a,24,18,4,2,0,6,6,3,2,0,0.00
65890,2025-01-10,US,57,num01,b,32,26,5,1,0,9,9,5,1,0,0.00
65893,2025-01-10,US,151,num01,a,24,21,2,1,0,5,5,2,1,0,0.00
65902,2025-01-10,US,186,num01,a,5,5,0,0,0,2,2,0,0,0,0.00


In [8]:
(
    buckets.groupby('date_day')
    .total_unique_users.sum()
    .to_frame()
    .tail(20)
)
    

,total_unique_users
date_day,
2025-02-12,6242
2025-02-13,6667
2025-02-14,6517
2025-02-15,7371
2025-02-16,7180
2025-02-17,6293
2025-02-18,7224
2025-02-19,6829
2025-02-20,6435
